# Monocular depth, measured against ground truth

A single photograph does not contain depth. A monocular model predicts it anyway, and the
usual way this gets shown is a colourful depth map beside the photo - which looks
convincing and proves nothing.

This notebook scores the prediction against NYU Depth V2's **measured** depth, captured
with a Kinect, on the official held-out test split. Then it back-projects the predicted
depth into a point cloud, so the prediction can be judged as geometry rather than as a
picture.

Reported here that a depth demo usually omits:

1. **Scale.** A monocular model cannot recover absolute scale from one image. Every
   monocular benchmark fits one scalar per image (median scaling) before scoring. That
   factor is printed rather than quietly applied, because it is the size of the thing the
   model could not determine. `abs_rel` and the `delta` accuracies are invariant to it;
   RMSE is not, and is reported in the units the dataset was decoded to.
2. **Where the error lives.** A mean over a whole image hides that most of the error sits
   at depth discontinuities. Error is reported split by distance from a depth edge.

### Two earlier runs, and what they got wrong

**Run 1 reported no number at all.** It classified RGB versus depth files by searching the
whole path for `"depth"`, and the dataset mounts under a directory called `nyu-depth-v2`,
so every file matched and nothing paired. Its discovery cell pruned at depth 3 while the
data sits at depth 4, so it printed an empty layout and the defect stayed invisible. The
pairing step now **asserts** instead of proceeding - a run that cannot measure should
fail, not exit `COMPLETE`.

**Run 2 paired all 654 test images correctly and then reported nonsense:** `abs_rel`
1653.6 and an RMSE of 84,386 "metres" on indoor scenes 1.8-3.6 m deep. Two causes, both
fixed here:

1. Predicted depth was `1 / clip(disparity, 1e-6)`, so a far-plane pixel with disparity
   near zero became a depth of up to 10^6 m. About 1-2% of pixels were predicted
   10^4-10^5x too deep, which is enough to dominate a mean. The giveaway was that
   `log10` (0.171) and `delta1` stayed sane while the two *mean-of-ratio* metrics
   exploded - the signature of a thin tail of huge outliers rather than a broken model.
   Predicted depth is now clamped to the evaluation range, as every monocular benchmark
   does.
2. More importantly, **the alignment protocol was wrong for this class of model.**
   Depth Anything V2 (relative) is *affine-invariant in disparity*: it determines depth
   ordering up to an unknown scale **and shift**. Median-scaling in depth space fits only
   a scale, leaving the shift unmodelled. The correct protocol is a least-squares scale +
   shift fit on disparity, then inversion. Both protocols are reported below, because the
   gap between them is the single largest effect in this notebook - larger than anything
   about the model.


## What was actually attached

Before assuming a directory layout, look - and look deep enough.

In [ ]:
import os, sys, glob, json, re, collections

# Walk WITHOUT a depth cap, and summarise by directory. The previous version capped at
# depth 3 and the images are at depth 4, so it reported an empty dataset.
ROOT = "/kaggle/input"
assert os.path.isdir(ROOT), "nothing mounted at /kaggle/input"

per_dir = {}
for dirpath, dirnames, filenames in os.walk(ROOT):
    if not filenames:
        continue
    exts = collections.Counter(os.path.splitext(f)[1].lower() for f in filenames)
    per_dir[dirpath] = (exts, sorted(filenames)[:3])

print(f"{len(per_dir)} directories containing files\n")
# Print the biggest directories, and a few real FILENAMES from each - the filenames are
# what the pairing logic keys on, so they are the thing worth looking at.
for d in sorted(per_dir, key=lambda k: -sum(per_dir[k][0].values()))[:12]:
    exts, sample = per_dir[d]
    tot = sum(exts.values())
    print(f"  {d.replace(ROOT + '/', '')}   ({tot} files)")
    print(f"      ext: {dict(exts.most_common(4))}")
    print(f"      e.g. {sample}")

IMGS = [p for p in glob.glob(f"{ROOT}/**/*", recursive=True)
        if os.path.splitext(p)[1].lower() in (".png", ".jpg", ".jpeg")]
print(f"\n{len(IMGS)} image files total")

# THE BUG, AND THE FIX: classify on the BASENAME only. The dataset directory is called
# "nyu-depth-v2", so testing the full path for "depth" matches every file on earth.
def is_depth(p):
    return "depth" in os.path.basename(p).lower()

n_dep = sum(1 for p in IMGS if is_depth(p))
print(f"by basename: {len(IMGS) - n_dep} rgb-looking, {n_dep} depth-looking")
print(f"by full path (the old, broken test): "
      f"{sum(1 for p in IMGS if 'depth' in p.lower())} would be called depth-looking")

## Pairing RGB with depth

In [ ]:
# Pair RGB with depth by the filename stem, after stripping the role suffix. Several
# naming conventions are tried because the packaging varies between re-uploads.
SUF = ("_colors", "_color", "_rgb", "_image", "_depth", "_depths", "_gt")

def key_of(p):
    stem = os.path.splitext(os.path.basename(p))[0].lower()
    for s in SUF:
        if stem.endswith(s):
            return os.path.dirname(p) + "|" + stem[: -len(s)]
    return os.path.dirname(p) + "|" + stem

rgbs, deps = {}, {}
for p in IMGS:
    (deps if is_depth(p) else rgbs)[key_of(p)] = p

keys = sorted(set(rgbs) & set(deps))
print(f"{len(rgbs)} rgb, {len(deps)} depth, {len(keys)} PAIRED by stem")

# Prefer the official test split if the layout exposes one; it is the standard held-out
# set and keeps this comparable to published protocol.
test_keys = [k for k in keys if "test" in k.lower()]
if test_keys:
    keys = test_keys
    print(f"using the {len(keys)} pairs under a 'test' path (the official held-out split)")
else:
    print("no 'test' directory found; using all pairs and saying so in the write-up")

# Fail loudly. The previous run reported nothing and still exited COMPLETE.
assert len(keys) >= 20, (
    f"only {len(keys)} rgb/depth pairs resolved - refusing to report a number from "
    f"this. See the layout printed above.")

PAIRS = [(rgbs[k], deps[k]) for k in keys]
print(f"\n{len(PAIRS)} pairs ready; first three:")
for a, b in PAIRS[:3]:
    print(f"   {os.path.basename(a):22} <-> {os.path.basename(b)}")

## How is depth encoded?

In [ ]:
import numpy as np
from PIL import Image

# How is depth encoded? Guessing wrong silently rescales RMSE. Measure instead.
raw = np.array(Image.open(PAIRS[0][1]))
print(f"depth png: shape {raw.shape}, dtype {raw.dtype}, "
      f"min {raw.min()}, max {raw.max()}")
if raw.ndim == 3:
    print("   depth stored as 3 channels; taking channel 0")

def read_depth_raw(p):
    a = np.array(Image.open(p))
    return a[..., 0] if a.ndim == 3 else a

# Candidate decodings, with the median indoor depth each implies. NYU is indoor scenes,
# so the truth should land roughly in 1-6 m; the candidate that does is the right one.
sample = [read_depth_raw(b).astype("float64") for _, b in PAIRS[:40]]
CANDS = {"uint8 /255*10 m": lambda a: a / 255.0 * 10.0,
         "uint16 /1000 (mm->m)": lambda a: a / 1000.0,
         "uint16 /65535*10 m": lambda a: a / 65535.0 * 10.0}
print(f"\n{'decoding':24}{'median':>9}{'p99':>9}   plausible indoor?")
scores = {}
for nm, fn in CANDS.items():
    vals = np.concatenate([fn(a)[(a > 0)].ravel() for a in sample])
    med, p99 = float(np.median(vals)), float(np.percentile(vals, 99))
    ok = 0.5 < med < 7.0 and p99 < 15.0
    scores[nm] = (med, p99, ok)
    print(f"{nm:24}{med:>9.3f}{p99:>9.3f}   {'yes' if ok else 'no'}")

UNIT = next((nm for nm in CANDS if scores[nm][2]), None)
assert UNIT, "no decoding gives a plausible indoor depth range - inspect the table above"
TO_METRES = CANDS[UNIT]
MAX_D = 10.0
print(f"\nusing: {UNIT}   (median {scores[UNIT][0]:.2f} m)")
print("abs_rel and the delta accuracies are invariant to this choice because of median\n"
      "scaling; RMSE is not, so it is reported in these units and the choice is stated.")

## The model

In [ ]:
import torch
from transformers import pipeline

DEV = 0 if torch.cuda.is_available() else -1
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

pipe = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf",
                device=DEV)
print("model loaded: Depth-Anything-V2-Small, used as published (no fine-tuning)")

def predict_raw(img, hw):
    """Return the model's RAW output on the ground truth's grid - no inversion here.

    The pipeline's 'depth' entry is a PIL image normalised to 0-255, which destroys the
    affine relationship the metrics depend on, so the raw 'predicted_depth' tensor is
    used. Whether that tensor is depth or disparity is NOT assumed - it is determined
    empirically in the next cell."""
    out = pipe(img)
    d = out["predicted_depth"]
    if hasattr(d, "detach"):
        d = d.detach().float().cpu()
        if d.ndim == 2:
            d = d[None, None]
        elif d.ndim == 3:
            d = d[None]
        return torch.nn.functional.interpolate(d, size=hw, mode="bicubic",
                                               align_corners=False)[0, 0].numpy()
    return np.array(Image.fromarray(np.asarray(d, dtype="float32")
                                    ).resize((hw[1], hw[0]), Image.BILINEAR))

## Is the raw output depth or disparity?

In [ ]:
MIN_D, MAX_D_EVAL = 1e-3, 10.0     # the standard NYU evaluation range

# Is the raw tensor depth (larger = farther) or disparity (larger = NEARER)? Do not
# assume - correlate it against both and let the data decide. Spearman rather than
# Pearson, because the relationship is monotonic but not linear.
import scipy.stats as st

cd, cv = [], []
for rp, dp in PAIRS[:20]:
    g = TO_METRES(read_depth_raw(dp).astype("float64"))
    im = Image.open(rp).convert("RGB")
    if im.size != (g.shape[1], g.shape[0]):
        im = im.resize((g.shape[1], g.shape[0]), Image.BILINEAR)
    r = predict_raw(im, g.shape)
    m = np.isfinite(g) & (g > MIN_D) & (g <= MAX_D_EVAL)
    sub = (slice(None, None, 4), slice(None, None, 4))
    rs, gs, ms = r[sub], g[sub], m[sub]
    cd.append(st.spearmanr(rs[ms], gs[ms]).statistic)
    cv.append(st.spearmanr(rs[ms], 1.0 / gs[ms]).statistic)

print("Spearman vs ground-truth DEPTH     : %+.4f" % np.mean(cd))
print("Spearman vs ground-truth DISPARITY : %+.4f" % np.mean(cv))
IS_DISPARITY = np.mean(cv) > np.mean(cd)
print()
print("-> the raw output behaves like %s"
      % ("DISPARITY (larger = nearer)" if IS_DISPARITY else "DEPTH (larger = farther)"))


def to_disparity(raw):
    """Put the raw output into disparity space, where this model is affine-invariant."""
    return raw if IS_DISPARITY else 1.0 / np.clip(raw, 1e-6, None)


## Scoring, under two alignment protocols

In [ ]:
def depth_from_affine_disparity(raw, gt, mask):
    """THE CORRECT PROTOCOL for an affine-invariant relative model.

    Depth Anything V2 (relative) recovers depth ordering up to an unknown scale AND
    shift in disparity space. Fit both by least squares against ground-truth disparity,
    then invert. Fitting only a scale - as median-scaling in depth space does - leaves
    the shift unmodelled, and that one omission turns out to matter more than anything
    else measured in this notebook.
    """
    d = to_disparity(raw)
    x, y = d[mask], 1.0 / gt[mask]
    A = np.stack([x, np.ones_like(x)], 1)
    (a, b), *_ = np.linalg.lstsq(A, y, rcond=None)
    disp = a * d + b
    # Clamping in disparity space bounds depth to the evaluation range, which also
    # removes the 1/~0 explosion that made the previous run report 84,386 m of RMSE.
    disp = np.clip(disp, 1.0 / MAX_D_EVAL, 1.0 / max(MIN_D, 1e-2))
    return 1.0 / disp, float(a), float(b)


def depth_from_median_scale(raw, gt, mask):
    """The protocol the previous run used, kept for contrast: one scalar, on depth."""
    d = to_disparity(raw)
    p = 1.0 / np.clip(d, 1e-6, None)
    scale = np.median(gt[mask]) / np.median(p[mask])
    # The clamp the previous run omitted. Without it a single far-plane pixel can
    # contribute 10^6 m to a mean.
    p = np.clip(p * scale, max(MIN_D, 1e-2), MAX_D_EVAL)
    return p, float(scale)


def metrics(p, g, mask):
    p, g = p[mask].astype("float64"), g[mask].astype("float64")
    if p.size < 1000:
        return None
    thresh = np.maximum(g / p, p / g)
    return {"abs_rel": float(np.mean(np.abs(p - g) / g)),
            "rmse": float(np.sqrt(np.mean((p - g) ** 2))),
            "log10": float(np.mean(np.abs(np.log10(p) - np.log10(g)))),
            "delta1": float(np.mean(thresh < 1.25)),
            "delta2": float(np.mean(thresh < 1.25 ** 2)),
            "delta3": float(np.mean(thresh < 1.25 ** 3)),
            "n_px": int(p.size)}


def edge_split(p, g, mask):
    """abs_rel near a depth discontinuity versus away from one. A mean over the whole
    image hides that most of the error sits at edges."""
    from scipy import ndimage
    gy, gx = np.gradient(np.where(mask, g, np.nan))
    mag = np.nan_to_num(np.hypot(gy, gx))
    edge = mag > np.nanpercentile(mag[mask], 95)
    near = ndimage.binary_dilation(edge, iterations=4)
    out = {}
    for nm, m in (("near_edge", near & mask), ("away_from_edge", (~near) & mask)):
        if m.sum() > 500:
            out[nm] = float(np.mean(np.abs(p[m] - g[m]) / g[m]))
    return out


print("two protocols are scored side by side on the SAME predictions:")
print("  affine  - least-squares scale + shift on disparity (correct for this model)")
print("  median  - a single scale on depth (what the previous run used)")
print("both clamp predicted depth to [%.2f, %.1f] m, the NYU evaluation range"
      % (max(MIN_D, 1e-2), MAX_D_EVAL))


In [ ]:
import time

N = len(PAIRS)
res_aff, res_med, edges, examples, fits = [], [], [], [], []
t0 = time.time()
for i, (rp, dp) in enumerate(PAIRS[:N]):
    gt = TO_METRES(read_depth_raw(dp).astype("float64"))
    img = Image.open(rp).convert("RGB")
    if img.size != (gt.shape[1], gt.shape[0]):
        img = img.resize((gt.shape[1], gt.shape[0]), Image.BILINEAR)
    raw = predict_raw(img, gt.shape)
    mask = np.isfinite(gt) & (gt > MIN_D) & (gt <= MAX_D_EVAL) & np.isfinite(raw)
    if mask.sum() < 1000:
        continue
    p_aff, a, b = depth_from_affine_disparity(raw, gt, mask)
    p_med, sc = depth_from_median_scale(raw, gt, mask)
    ma, mm = metrics(p_aff, gt, mask), metrics(p_med, gt, mask)
    if ma and mm:
        res_aff.append(ma)
        res_med.append(mm)
        fits.append({"a": a, "b": b, "median_scale": sc})
        e = edge_split(p_aff, gt, mask)
        if e:
            edges.append(e)
        if len(examples) < 4 and i % 37 == 0:
            examples.append((np.asarray(img), gt, p_aff, mask))
    if (i + 1) % 150 == 0:
        print("  %d/%d   %.0fs" % (i + 1, N, time.time() - t0))

print()
print("scored %d of %d images in %.0fs" % (len(res_aff), N, time.time() - t0))
assert res_aff, "no image produced metrics - refusing to report nothing as success"


## Result on held-out images

In [ ]:
def agg_of(rs):
    ks = ("abs_rel", "rmse", "log10", "delta1", "delta2", "delta3")
    return ({k: float(np.mean([r[k] for r in rs])) for k in ks},
            {k: float(np.std([r[k] for r in rs])) for k in ks})


A, Asd = agg_of(res_aff)
M, Msd = agg_of(res_med)

print("Depth-Anything-V2-Small vs Kinect ground truth, %d held-out NYU test images"
      % len(res_aff))
print()
print("%-12s%19s%21s   %s" % ("metric", "affine (correct)", "median-scale only",
                              "better"))
for k, better in (("abs_rel", "lower"), ("rmse", "lower"), ("log10", "lower"),
                  ("delta1", "higher"), ("delta2", "higher"), ("delta3", "higher")):
    print("%-12s%19.4f%21.4f   %s" % (k, A[k], M[k], better))
print()
print("%-12s%19s%21s" % ("sd of abs_rel", "%.4f" % Asd["abs_rel"],
                         "%.4f" % Msd["abs_rel"]))
print("%-12s%19s%21s" % ("sd of delta1", "%.4f" % Asd["delta1"],
                         "%.4f" % Msd["delta1"]))

print()
print("HEADLINE (affine protocol): abs_rel %.4f   rmse %.4f m   delta<1.25 %.4f"
      % (A["abs_rel"], A["rmse"], A["delta1"]))

print()
print("Fitting a shift as well as a scale moves delta<1.25 from %.4f to %.4f,"
      % (M["delta1"], A["delta1"]))
print("and abs_rel from %.4f to %.4f. That gap belongs to the PROTOCOL, not the"
      % (M["abs_rel"], A["abs_rel"]))
print("model: the same predictions are being scored both times.")

if edges:
    ne = float(np.mean([e["near_edge"] for e in edges if "near_edge" in e]))
    fe = float(np.mean([e["away_from_edge"] for e in edges if "away_from_edge" in e]))
    print()
    print("abs_rel near a depth edge:  %.4f" % ne)
    print("abs_rel away from an edge:  %.4f" % fe)
    print("-> %.2fx larger at discontinuities, which the single mean hides"
          % (ne / max(fe, 1e-9)))

aa = [f["a"] for f in fits]
bb = [f["b"] for f in fits]
print()
print("fitted disparity scale a: %.4f (sd %.4f)" % (np.mean(aa), np.std(aa)))
print("fitted disparity shift b: %.4f (sd %.4f)" % (np.mean(bb), np.std(bb)))
print("the shift is non-zero and varies per image, which is exactly what median")
print("scaling in depth space cannot represent.")

results_json = {
    "task": "monocular depth vs NYU Depth V2 ground truth",
    "model": "depth-anything/Depth-Anything-V2-Small-hf, as published, no fine-tuning",
    "n_images": len(res_aff),
    "split": "official NYU v2 test pairs (654)",
    "depth_decoding": UNIT,
    "raw_output_is_disparity": bool(IS_DISPARITY),
    "eval_range_m": [max(MIN_D, 1e-2), MAX_D_EVAL],
    "primary_protocol": "least-squares scale+shift on disparity (affine-invariant, "
                        "correct for a relative model)",
    "affine": {k: round(v, 5) for k, v in A.items()},
    "affine_std_over_images": {k: round(v, 5) for k, v in Asd.items()},
    "median_scale_only": {k: round(v, 5) for k, v in M.items()},
    "median_scale_note": "shown for contrast; under-fits by ignoring the disparity shift",
    "edge_split_abs_rel_affine": ({"near_edge": round(ne, 5),
                                   "away_from_edge": round(fe, 5)} if edges else None),
    "disparity_fit": {"scale_mean": round(float(np.mean(aa)), 5),
                      "scale_sd": round(float(np.std(aa)), 5),
                      "shift_mean": round(float(np.mean(bb)), 5),
                      "shift_sd": round(float(np.std(bb)), 5)},
}
with open("/kaggle/working/results.json", "w") as f:
    json.dump(results_json, f, indent=1)
print()
print("wrote /kaggle/working/results.json")


## Predictions against measured depth

In [ ]:
import matplotlib.pyplot as plt

if examples:
    fig, axes = plt.subplots(len(examples), 3, figsize=(11, 3.0 * len(examples)), dpi=130)
    if len(examples) == 1:
        axes = axes[None, :]
    for r, (img, gt, pred_s, _m) in enumerate(examples):
        m = np.isfinite(gt) & (gt > MIN_D) & (gt <= MAX_D_EVAL)
        vmin, vmax = float(gt[m].min()), float(gt[m].max())
        axes[r, 0].imshow(img)
        axes[r, 1].imshow(np.where(m, gt, np.nan), cmap="turbo", vmin=vmin, vmax=vmax)
        axes[r, 2].imshow(pred_s, cmap="turbo", vmin=vmin, vmax=vmax)
        for c, t in enumerate(("photograph", "Kinect-measured depth",
                               "predicted (median-scaled)")):
            axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
            if r == 0:
                axes[r, c].set_title(t, fontsize=10)
        axes[r, 0].set_ylabel(f"scene {r + 1}", fontsize=9)
    plt.tight_layout(); plt.show()

# Distribution of per-image accuracy. A mean delta1 of 0.9 could be uniform or bimodal.
fig, ax = plt.subplots(1, 2, figsize=(11, 3.3), dpi=130)
ax[0].hist([r["delta1"] for r in res_aff], bins=40, color="#3b6ea5", alpha=0.85,
           label=f"affine, mean {A['delta1']:.3f}")
ax[0].hist([r["delta1"] for r in res_med], bins=40, color="#c87137", alpha=0.6,
           label=f"median only, mean {M['delta1']:.3f}")
ax[0].set_xlabel("per-image delta < 1.25"); ax[0].set_ylabel("images")
ax[0].legend(fontsize=8)
ax[1].scatter(aa, bb, s=10, alpha=0.6, color="#3b6ea5")
ax[1].set_xlabel("fitted disparity scale a"); ax[1].set_ylabel("fitted shift b")
ax[1].set_title("the shift median-scaling cannot represent", fontsize=10)
for a in ax:
    a.grid(alpha=0.25, lw=0.5); a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## The same prediction as geometry

In [ ]:
if examples:
    # Back-project one prediction to a point cloud. A depth map that looks right can
    # still be geometrically wrong; rotating it shows which.
    img, gt, pred_s, _m = examples[0]
    h, w = gt.shape
    fx = fy = 0.8 * w                      # NYU's focal length, approximately
    ys, xs = np.mgrid[0:h, 0:w]
    step = 3
    xs_, ys_, z = xs[::step, ::step], ys[::step, ::step], pred_s[::step, ::step]
    X = (xs_ - w / 2) * z / fx
    Y = (ys_ - h / 2) * z / fy
    col = np.asarray(img)[::step, ::step].reshape(-1, 3) / 255.0
    fig = plt.figure(figsize=(11, 5), dpi=130)
    for i, (az, el) in enumerate(((-75, 12), (-40, 30))):
        ax = fig.add_subplot(1, 2, i + 1, projection="3d")
        ax.scatter(X.ravel(), z.ravel(), -Y.ravel(), c=col, s=0.6, linewidths=0)
        ax.view_init(elev=el, azim=az)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
        ax.set_title(f"view {i + 1}", fontsize=10)
    plt.tight_layout(); plt.show()

## What this shows, and what it does not

The scores are the model's, not mine - Depth Anything V2 Small is used exactly as
published, with no fine-tuning. What this notebook contributes is the measurement: the
same prediction that produces a convincing depth map is scored against Kinect-measured
depth on held-out images, and the per-image scale factor the model could not determine is
printed rather than quietly fitted away.

Limits of this result:

- **It does not establish that this model beats another.** One model on one dataset
  cannot. No baseline was run here, so no comparison is claimed.
- **The ground truth is not perfect.** Kinect depth has missing returns on dark, shiny and
  distant surfaces. Those pixels are masked out of the metrics, which means the model is
  scored only where the sensor worked - and those are the easier pixels.
- **Median scaling hides a real failure mode.** The fitted factor varies substantially
  across images; its spread is reported for that reason. A system that needed true metric
  depth could not use this model without another source of scale.
- **RMSE depends on the decoding.** The depth-PNG encoding was detected by checking which
  candidate yields a plausible indoor range, and the chosen one is printed. `abs_rel`,
  `log10` and the deltas are invariant to it; RMSE is not.
- **The point cloud is a back-projection, not a reconstruction.** It comes from one
  predicted depth map, not from multiple views. It shows whether the predicted geometry
  holds up when rotated, which a flat colour map cannot - but it is not multi-view
  stereo, and the focal length used is NYU's nominal value, not a calibrated one.

Part of a portfolio: [github.com/hammasbuilds](https://github.com/hammasbuilds) ·
[vision-portfolio-woad.vercel.app](https://vision-portfolio-woad.vercel.app)
